In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Carrega os dados processados na Etapa 2
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()
arm_train = pd.read_csv("../data/processed/arm_train.csv").squeeze()
arm_test = pd.read_csv("../data/processed/arm_test.csv").squeeze()

# Retreina os dois modelos por braço (mesma lógica da Etapa 3)
model_arm0 = LogisticRegression(max_iter=1000)
model_arm0.fit(X_train[arm_train == 0], y_train[arm_train == 0])

model_arm1 = LogisticRegression(max_iter=1000)
model_arm1.fit(X_train[arm_train == 1], y_train[arm_train == 1])

print("Modelo braço 0 (cellular) treinado com", (arm_train == 0).sum(), "linhas")
print("Modelo braço 1 (telephone) treinado com", (arm_train == 1).sum(), "linhas")

/home/dev/miniconda3/envs/tc5/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Modelo braço 0 (cellular) treinado com 20908 linhas
Modelo braço 1 (telephone) treinado com 12042 linhas


/home/dev/miniconda3/envs/tc5/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [2]:
# Amostra 5 clientes do teste, de forma reprodutível
golden_set = X_test.sample(n=5, random_state=42)

# Probabilidade prevista de conversão em cada braço, para esses 5 clientes
p_arm0 = model_arm0.predict_proba(golden_set)[:, 1]
p_arm1 = model_arm1.predict_proba(golden_set)[:, 1]

# Braço recomendado = o de maior probabilidade prevista
recommended_arm = (p_arm1 > p_arm0).astype(int)  # 0 = cellular, 1 = telephone

# Dados reais desses clientes (braço usado no histórico e resultado real)
actual_arm = arm_test.loc[golden_set.index]
actual_outcome = y_test.loc[golden_set.index]

resumo = pd.DataFrame({
    "p_cellular": p_arm0.round(3),
    "p_telephone": p_arm1.round(3),
    "braco_recomendado": recommended_arm,
    "braco_real_historico": actual_arm.values,
    "resultado_real": actual_outcome.values,
})
resumo.index = golden_set.index
print(resumo)

      p_cellular  p_telephone  braco_recomendado  braco_real_historico  \
706        0.056        0.096                  1                     0   
5968       0.190        0.138                  0                     0   
1665       0.076        0.056                  0                     1   
6676       0.063        0.031                  0                     0   
5606       0.046        0.029                  0                     1   

      resultado_real  
706                0  
5968               0  
1665               0  
6676               0  
5606               0  


In [3]:
def justificativa(p0, p1, braco):
    if braco == 0:
        diff = (p0 - p1) * 100
        return f"Recomendado cellular: probabilidade prevista de conversão {p0*100:.1f}% vs {p1*100:.1f}% em telephone (+{diff:.1f}pp)."
    else:
        diff = (p1 - p0) * 100
        return f"Recomendado telephone: probabilidade prevista de conversão {p1*100:.1f}% vs {p0*100:.1f}% em cellular (+{diff:.1f}pp)."

resumo["justificativa"] = [
    justificativa(p0, p1, braco)
    for p0, p1, braco in zip(resumo["p_cellular"], resumo["p_telephone"], resumo["braco_recomendado"])
]

resumo["braco_recomendado_nome"] = resumo["braco_recomendado"].map({0: "cellular", 1: "telephone"})
resumo["braco_real_nome"] = resumo["braco_real_historico"].map({0: "cellular", 1: "telephone"})

golden_final = resumo[["braco_recomendado_nome", "justificativa", "braco_real_nome", "resultado_real"]]
golden_final.columns = ["oferta_recomendada", "justificativa", "canal_usado_historico", "converteu_de_fato"]

for idx, row in golden_final.iterrows():
    print(f"\nCliente {idx}")
    print(f"  Oferta recomendada: {row['oferta_recomendada']}")
    print(f"  Justificativa: {row['justificativa']}")
    print(f"  Canal usado no histórico: {row['canal_usado_historico']} | Converteu de fato: {'sim' if row['converteu_de_fato'] == 1 else 'não'}")

golden_final.to_csv("../data/processed/golden_set.csv")
print("\nSalvo em data/processed/golden_set.csv")


Cliente 706
  Oferta recomendada: telephone
  Justificativa: Recomendado telephone: probabilidade prevista de conversão 9.6% vs 5.6% em cellular (+4.0pp).
  Canal usado no histórico: cellular | Converteu de fato: não

Cliente 5968
  Oferta recomendada: cellular
  Justificativa: Recomendado cellular: probabilidade prevista de conversão 19.0% vs 13.8% em telephone (+5.2pp).
  Canal usado no histórico: cellular | Converteu de fato: não

Cliente 1665
  Oferta recomendada: cellular
  Justificativa: Recomendado cellular: probabilidade prevista de conversão 7.6% vs 5.6% em telephone (+2.0pp).
  Canal usado no histórico: telephone | Converteu de fato: não

Cliente 6676
  Oferta recomendada: cellular
  Justificativa: Recomendado cellular: probabilidade prevista de conversão 6.3% vs 3.1% em telephone (+3.2pp).
  Canal usado no histórico: cellular | Converteu de fato: não

Cliente 5606
  Oferta recomendada: cellular
  Justificativa: Recomendado cellular: probabilidade prevista de conversão 4.6% 